In [33]:
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)


# ==========================
# OpenAI Configuration
# ==========================

config_str = os.getenv("GPT_LUNA_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)

MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)

DATA_PATH = os.getenv("DATA_PATH", "../data/")

df=pd.read_csv(f'{DATA_PATH}df_n3.csv')



Using model: gpt-5.6-luna with reasoning effort: low


In [ ]:
import json
import time
import random
import pandas as pd
from collections import defaultdict
from collections import Counter


# ==========================
# Global token usage
# ==========================

TOTAL_USAGE = {
    "prompt_tokens": 0,
    "reasoning_tokens": 0,
    "total_tokens": 0,
    "num_calls": 0,
}


def update_usage(usage):
    """
    Accumulate token usage across all LLM calls.
    """
    global TOTAL_USAGE

    u = usage.model_dump()

    TOTAL_USAGE["prompt_tokens"] += u.get("prompt_tokens", 0)
    TOTAL_USAGE["total_tokens"] += u.get("total_tokens", 0)
    TOTAL_USAGE["reasoning_tokens"] += (
        u.get("completion_tokens_details", {})
         .get("reasoning_tokens", 0)
    )
    TOTAL_USAGE["num_calls"] += 1


N_CONSISTENCY_RUNS = 3


# =========================
# PROMPT BUILDING
# =========================

def build_dedup_messages(rows, shuffle=False):
    """
    rows: list of dicts for comments that share the SAME patch_id.
          each dict must have: comment_id, generated_comment, 
        #   category, severity,
          hunk, generated_context
    """
    code_patch = rows[0]
    comments = list(rows)
    if shuffle:
        random.shuffle(comments)

    comments_text = "\n\n".join(
        f'ID: {c["comment_id"]}\n'
        f'Comment: {c["generated_comment"]}'
        # f'Category: {c["category"]}\n'
        # f'Severity: {c["severity"]}\n'
        for c in comments
    )

    system_prompt = """
You are an expert software engineer reviewing a set of code-review comments that were independently generated by different models for the SAME code change.

INPUT:
You may receive:
- Pull request title
- Target file
- Related code hunks from the same file
- Surrounding code context, wrapped in <context> tags
- The code patch, wrapped in <patch> tags
- A list of generated review comments, each with its ID, category, and severity

Use the available context only to understand the patch and determine whether two comments refer to the same underlying issue. Do not group comments simply because they mention nearby code or similar identifiers.

TASK 1 - GROUPING:
Group comments that describe the SAME underlying issue, even if phrased differently or focused on different symptoms. Two comments belong in the same group ONLY IF a human reviewer would consider one redundant given the other.

Do NOT group comments together just because they:
- refer to the same lines, variables, or functions but raise different concerns (e.g. bug vs. refactoring suggestion).
- use similar wording but identify different root causes.

Every comment must appear in exactly one group. A comment with no duplicate forms its own group.

TASK 2 - SYNTHESIS:
For every group containing TWO OR MORE comments, generate a single "synthesis_comment" that represents their shared concern. It should:
- Capture only the common issue.
- Be concise.
- Read like a human review comment, not a summary of comments.
- Avoid copying any individual comment verbatim.

Do NOT include "synthesis_comment" for singleton groups.

OUTPUT (STRICT JSON ONLY):
{
  "groups": [
    {
      "comment_ids": ["<id1>", "<id2>"],
      "issue_type": "short label",
      "target": "variable/function/line",
      "synthesis_comment": "..."
    },
    {
      "comment_ids": ["<id3>"],
      "issue_type": "short label",
      "target": "..."
    }
  ]
}

Use ONLY the exact comment IDs provided. Do not invent, split, merge, or omit IDs.
"""
    user_prompt = ""

    if "pr_title" in code_patch and pd.notna(code_patch["pr_title"]):

        user_prompt += f"""
PULL REQUEST TITLE:
{code_patch["pr_title"]}

"""

    if "target_file" in code_patch and pd.notna(code_patch["target_file"]):

        user_prompt += f"""
TARGET FILE:
{code_patch["target_file"]}

"""
    if "relevant_same_file_code_hunks" in code_patch and pd.notna(code_patch["relevant_same_file_code_hunks"]):

        user_prompt += f"""
RELATED CODE HUNKS IN TARGET FILE:
{code_patch["relevant_same_file_code_hunks"]}

"""

    if "relevant_context" in code_patch and pd.notna(code_patch["relevant_context"]):

        if str(code_patch["relevant_context"]).strip():

            user_prompt += """
SURROUNDING CODE CONTEXT:
This is additional context extracted from the original file around the changed code to help you in problem identification.
<context>
"""

            user_prompt += code_patch["relevant_context"]
            user_prompt += "\n</context>\n\n"

    user_prompt += f"""
CODE PATCH:
<patch>
{code_patch["hunk"]}
</patch>

"""
    user_prompt += f"""
COMMENTS TO GROUP:
{comments_text}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


# =========================
# PARSING + VALIDATION
# =========================

def parse_dedup_output(text, valid_ids):
    """
    Returns list of group dicts:
        {"comment_ids": [...], "synthesis_comment": <str or None>}
    validated against valid_ids, or None if the JSON couldn't be parsed at
    all (caller retries).

    - hallucinated / unknown ids are dropped
    - ids the model forgot to place anywhere become their own singleton
      group (synthesis_comment = None)
    - ids the model duplicated across groups are kept only in the first
      group
    - synthesis_comment is dropped (set to None) for any group that ends up
      with only one comment_id after cleaning, even if the model provided one
    """
    if text is None:
        return None

    try:
        data = json.loads(text)
        raw_groups = data.get("groups", [])
    except Exception:
        return None

    valid_ids = set(valid_ids)
    seen = set()
    groups = []

    for g in raw_groups:
        ids = g.get("comment_ids", [])
        clean_ids = [i for i in ids if i in valid_ids and i not in seen]
        if not clean_ids:
            continue
        seen.update(clean_ids)

        synthesis = g.get("synthesis_comment") if len(clean_ids) > 1 else None
        if isinstance(synthesis, str):
            synthesis = synthesis.strip() or None

        groups.append({"comment_ids": clean_ids,
                      "synthesis_comment": synthesis})

    missing = valid_ids - seen
    for m in missing:
        groups.append({"comment_ids": [m], "synthesis_comment": None})

    return groups


def groups_to_pairs(groups):
    """Set of frozenset({id_a, id_b}) pairs that co-occurred in the same
    group within a single run.

    groups: list of dicts {"comment_ids": [...], "synthesis_comment": ...}
    """
    pairs = set()
    for g in groups:
        ids = g["comment_ids"]
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                pairs.add(frozenset((ids[i], ids[j])))
    return pairs


class UnionFind:
    def __init__(self, ids):
        self.parent = {i: i for i in ids}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


def select_synthesis_for_cluster(cluster_ids, all_run_groups):
    """
    Pick the best synthesis_comment for a final (post union-find) cluster.

    cluster_ids: list of comment_ids in the final cluster (len > 1)
    all_run_groups: list of raw group dicts (with synthesis_comment) pooled
                     across every consistency run for this patch.

    Strategy: among raw groups that have a synthesis_comment, prefer an
    exact id-set match; otherwise fall back to the highest-overlap
    (Jaccard) raw group so we still reuse model-authored text instead of
    making a fresh LLM call.
    """
    target = set(cluster_ids)
    best_group = None
    best_score = -1.0

    for g in all_run_groups:
        if not g.get("synthesis_comment"):
            continue
        gid_set = set(g["comment_ids"])
        if gid_set == target:
            return g["synthesis_comment"]  # exact match, best possible

        intersection = len(gid_set & target)
        if intersection == 0:
            continue
        union_size = len(gid_set | target)
        jaccard = intersection / union_size
        if jaccard > best_score:
            best_score = jaccard
            best_group = g

    return best_group["synthesis_comment"] if best_group else None


# =========================
# LLM CALL
# =========================

def call_llm(messages, temperature=0.4):
    kwargs = {
        "model": MODEL_NAME,
        "messages": messages,
        "timeout": 120,
    }

    if REASONING_EFFORT:
        kwargs["reasoning_effort"] = REASONING_EFFORT
    else:
        kwargs["temperature"] = temperature

    response = client.chat.completions.create(**kwargs)

    update_usage(response.usage)

    raw_text = response.choices[0].message.content

    return raw_text


def call_llm_with_retries(messages, max_attempts=3, base_sleep=1.5):
    sleep_time = base_sleep

    for attempt in range(max_attempts):
        try:
            return call_llm(messages)

        except Exception as e:
            print(f"[ERROR] {e}")

            if attempt == max_attempts - 1:
                print('max attempts reached, giving up')
                break

            print(f'retrying, attempt nb: {attempt}')
            wait = sleep_time + random.uniform(0, 1)
            print(f"[BACKOFF] sleeping {wait:.2f}s")
            time.sleep(wait)
            sleep_time *= 2

    return None


# =========================
# SELF-CONSISTENCY: 3 RUNS -> INTERSECTION -> UNION-FIND
# =========================


def run_dedup_for_patch(patch_id, rows, n_runs=N_CONSISTENCY_RUNS):
    """
    Returns a FLAT LIST of cluster dicts (one dict per cluster) for this
    patch - not a single nested patch-level object.
    """
    valid_ids = [r["comment_id"] for r in rows]

    if len(rows) <= 1:
        clusters = [[r["comment_id"]] for r in rows]
        return build_cluster_rows(patch_id, clusters, rows, all_run_groups=[], pair_agreements={})

    run_pair_sets = []
    all_run_groups = []  # pooled raw groups (with synthesis) across all runs

    for run_idx in range(n_runs):
        # shuffle order on repeat runs (run 0 keeps dataset order) so we can
        # detect groupings that are just a position-bias artifact
        messages = build_dedup_messages(rows, shuffle=(run_idx > 0))
        raw_text = call_llm_with_retries(messages)
        groups = parse_dedup_output(raw_text, valid_ids)

        if groups is None:
            print(f"[WARN] patch {patch_id} run {run_idx}: unparseable output, "
                  f"treating all comments as singletons for this run")
            print(f"raw_text: {raw_text}  ")
            groups = [{"comment_ids": [i], "synthesis_comment": None}
                      for i in valid_ids]

        run_pair_sets.append(groups_to_pairs(groups))
        all_run_groups.extend(groups)
        time.sleep(
            SLEEP_BETWEEN_CALLS if "SLEEP_BETWEEN_CALLS" in globals() else 1.0)

    # Majority agreement threshold
    min_votes = len(run_pair_sets) // 2 + 1

    pair_counts = Counter()

    for pairs in run_pair_sets:
        for pair in pairs:
            pair_counts[pair] += 1

    # Keep pairs that appeared in at least majority of runs
    pair_agreements = {
        pair: {
            "agreement_count": count,
            "agreement_ratio": count / len(run_pair_sets)
        }
        for pair, count in pair_counts.items()
    }

    consistent_pairs = {
        pair
        for pair, info in pair_agreements.items()
        if info["agreement_count"] >= min_votes
    }

    uf = UnionFind(valid_ids)
    for pair in consistent_pairs:
        a, b = tuple(pair)
        uf.union(a, b)

    clusters_map = defaultdict(list)
    for i in valid_ids:
        clusters_map[uf.find(i)].append(i)

    return build_cluster_rows(
        patch_id,
        list(clusters_map.values()),
        rows,
        all_run_groups,
        pair_agreements
    )


def get_cluster_agreement(cluster_ids, pair_agreements):

    agreements = []

    for i in range(len(cluster_ids)):
        for j in range(i + 1, len(cluster_ids)):

            pair = frozenset(
                [
                    cluster_ids[i],
                    cluster_ids[j]
                ]
            )

            if pair in pair_agreements:
                agreements.append(
                    pair_agreements[pair]["agreement_ratio"]
                )

    return agreements


def build_cluster_rows(patch_id, clusters, rows, all_run_groups, pair_agreements):
    """
    Builds ONE FLAT DICT PER CLUSTER (patch_id/num_comments folded into
    each row so no information is lost when exploding to per-cluster
    instances). Returns a list of these flat dicts.
    """
    lookup = {r["comment_id"]: r for r in rows}
    cluster_rows = []

    for k, ids in enumerate(clusters):
        synthesis = select_synthesis_for_cluster(
            ids, all_run_groups) if len(ids) > 1 else None

        cluster_rows.append({
            "patch_id": patch_id,
            "num_comments": len(ids),
            "cluster_id": f"{patch_id}_c{k}",
            "comment_ids": ids,
            "comments": [lookup[i]["generated_comment"] for i in ids],
            "generation_systems": [lookup[i]["generation_system"] for i in ids],
            "categories": [lookup[i]["category"] for i in ids],
            "severities": [lookup[i]["severity"] for i in ids],
            "synthesis_comment": synthesis,
            "agreement_scores": get_cluster_agreement(
                ids,
                pair_agreements
            ),
        })

    return cluster_rows


# =========================
# PIPELINE OVER FULL DATAFRAME
# =========================

def run_dedup_pipeline(df, checkpoint_every=10, checkpoint_path="artifacts/dedup_checkpoint.json"):
    """
    Returns a FLAT LIST where every element is a single cluster instance
    (one row per cluster, not one row per patch). `checkpoint_every` still
    counts by PATCHES processed, not clusters, so behavior/frequency of
    checkpointing stays the same as before.
    """
    required_cols = ["comment_id", "generated_comment", "category",
                     "generation_system", "patch_id", "severity", "hunk"]
    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    results = []  # flat list of cluster dicts across ALL patches
    patch_groups = df.groupby("patch_id")
    total = patch_groups.ngroups

    print(f"[START] Deduplicating comments across {total} patches")

    for i, (patch_id, group) in enumerate(patch_groups, 1):
        print(f"\n[PATCH {i}/{total}] {patch_id} ({len(group)} comments)")

        rows = group.to_dict("records")
        for r in rows:
            r.setdefault("generated_context", "")

        try:
            patch_cluster_rows = run_dedup_for_patch(patch_id, rows)
        except Exception as e:
            print(f"[FAIL] patch {patch_id}: {e}")
            patch_cluster_rows = [{
                "patch_id": patch_id,
                "num_comments": len(rows),
                "cluster_id": f"{patch_id}_c{k}",
                "comment_ids": [r["comment_id"]],
                "comments": [r["generated_comment"]],
                "generation_systems": [r["generation_system"]],
                "categories": [r["category"]],
                "severities": [r["severity"]],
                "synthesis_comment": None,
                "error": str(e),
            } for k, r in enumerate(rows)]

        results.extend(patch_cluster_rows)

        if i % checkpoint_every == 0:
            with open(checkpoint_path, "w") as f:
                json.dump(results, f, indent=2)
            print(f"[CHECKPOINT] saved {checkpoint_path}")

    with open(checkpoint_path, "w") as f:
        json.dump(results, f, indent=2)

    print("\n[DONE] Deduplication finished")
    print(
        "[DONE] Finished pipeline"
    )


    return pd.DataFrame(results)

In [ ]:
results=run_dedup_pipeline(df)
results.to_csv(f'{DATA_PATH}df_n4.csv', index=False)

[START] Deduplicating comments across 7 patches

[PATCH 1/7] P000001 (1 comments)

[PATCH 2/7] P000002 (1 comments)

[PATCH 3/7] P000004 (1 comments)

[PATCH 4/7] P000005 (1 comments)

[PATCH 5/7] P000006 (1 comments)

[PATCH 6/7] P000007 (2 comments)

[PATCH 7/7] P000008 (2 comments)

[DONE] Deduplication finished
[DONE] Finished pipeline


In [36]:
import json
from datetime import datetime
from pathlib import Path


def save_token_usage_log(
    task_name="comment_deduplication",
    log_file="../logs/token_usage_insights_logs.json"
):

    usage_stats = {
        "task": task_name,
        "timestamp": datetime.now().isoformat(),
        "model_name": MODEL_NAME,
        "num_calls": TOTAL_USAGE["num_calls"],
        "prompt_tokens": TOTAL_USAGE["prompt_tokens"],
        "reasoning_tokens": TOTAL_USAGE["reasoning_tokens"],
        "total_tokens": TOTAL_USAGE["total_tokens"],
    }

    log_path = Path(log_file)

    log_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        with open(log_path, "r") as f:
            logs = json.load(f)

    except (FileNotFoundError, json.JSONDecodeError):
        logs = []

    logs.append(usage_stats)

    with open(log_path, "w") as f:
        json.dump(
            logs,
            f,
            indent=4
        )

    print(f"Token usage saved to {log_path}")
    
save_token_usage_log()    

Token usage saved to ..\logs\token_usage_insights_logs.json
